# Clasificador de Patologías Estructurales en Hormigón Armado

Modelo XGBoost entrenado con dataset sintético de observaciones de campo.

**Pipeline:** Carga → Preprocesamiento → Split → Entrenamiento base

In [ ]:
# ============================================================
# Celda 1: Importar librerías necesarias
# Se importan todas las herramientas que usaremos en el notebook.
# ============================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Librerías cargadas correctamente.")

In [ ]:
# ============================================================
# Celda 2: Cargar el dataset sintético
# El CSV fue generado por generar_dataset_sintetico.py con
# 2216 observaciones de campo simuladas (96 patologías).
# ============================================================

df = pd.read_csv("dataset_patologias_sintetico.csv")

print(f"Dimensiones del dataset: {df.shape}")
print(f"Patologías únicas: {df['defecto_numero'].nunique()}")
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# ============================================================
# Celda 3: Separar features (X) y target (y)
# Todas las columnas son categóricas excepto defecto_numero,
# que es la variable objetivo (1 a 96).
# XGBoost necesita clases desde 0, así que restamos 1.
# Guardamos el mapeo para reconvertir después.
# ============================================================

X = df.drop(columns=["defecto_numero"])
y = df["defecto_numero"] - 1  # Convertir a 0-based (0 a 95)

columnas_categoricas = X.columns.tolist()

print(f"Features ({len(columnas_categoricas)}): {columnas_categoricas}")
print(f"Target: defecto_numero convertido a 0-based (clases {y.min()} a {y.max()})")

In [ ]:
# ============================================================
# Celda 4: Preprocesamiento con ColumnTransformer
# Se usa OneHotEncoder dentro de un ColumnTransformer para
# convertir todas las variables categóricas a numéricas.
# handle_unknown='ignore' permite que en predicción futura
# no falle si aparece un valor no visto en entrenamiento.
# ============================================================

preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), columnas_categoricas)
    ]
)

print("Preprocesador definido: OneHotEncoder para todas las features categóricas.")
print(f"Se transformarán {len(columnas_categoricas)} columnas.")

In [ ]:
# ============================================================
# Celda 5: Split train/test 80/20 con stratify
# stratify=y asegura que cada clase tenga la misma proporción
# en train y test, importante con 96 clases desbalanceadas.
# random_state=42 para reproducibilidad.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nClases en train: {y_train.nunique()} | Clases en test: {y_test.nunique()}")
print(f"\nDistribución en train (resumen):")
print(y_train.value_counts().describe())

In [ ]:
# ============================================================
# Celda 6: Pipeline completo (preprocesamiento + XGBoost base)
# Se encapsula todo en un Pipeline de sklearn para que el
# preprocesamiento y el modelo sean un solo objeto reutilizable.
# Modelo base con parámetros razonables sin optimizar aún.
# No se fija num_class: XGBoost lo infiere de y automáticamente.
# ============================================================

modelo_base = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("clasificador", XGBClassifier(
        n_estimators=200,
        eta=0.1,
        max_depth=6,
        gamma=0.5,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        verbosity=0
    ))
])

print("Pipeline definido:")
print(modelo_base)

In [ ]:
# ============================================================
# Celda 7: Entrenar el modelo base
# Entrenamos con los datos de train y evaluamos en train y test
# para tener una primera línea base antes de GridSearch.
# ============================================================

modelo_base.fit(X_train, y_train)

# Predicciones en train y test
y_pred_train = modelo_base.predict(X_train)
y_pred_test = modelo_base.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

print(f"Accuracy en TRAIN: {acc_train:.4f}")
print(f"Accuracy en TEST:  {acc_test:.4f}")
print(f"\nDiferencia (sobreajuste): {acc_train - acc_test:.4f}")
print("\n" + "="*50)
print("CLASSIFICATION REPORT (TEST):")
print("="*50)
print(classification_report(y_test, y_pred_test, zero_division=0))